# 6주차 — SELECT ④ GROUP BY·HAVING
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. GROUP BY

In [ ]:
# 진료과별 내원 건수와 환자 수 — 둘의 차이에 주목!
q("""
SELECT department, COUNT(*) AS 내원건수,
       COUNT(DISTINCT patient_id) AS 환자수
FROM visit GROUP BY department ORDER BY 내원건수 DESC
""")

In [ ]:
# 내원유형별 평균 재원일수
q("""
SELECT visit_type, COUNT(*) AS 건수,
       ROUND(AVG(julianday(discharge_date) - julianday(visit_date)), 1) AS 평균재원일
FROM visit GROUP BY visit_type
""")

In [ ]:
# 월별 내원 추이 (2025년)
q("""
SELECT strftime('%Y-%m', visit_date) AS 월, COUNT(*) AS 내원건수
FROM visit WHERE visit_date >= '2025-01-01'
GROUP BY strftime('%Y-%m', visit_date) ORDER BY 월
""")

## 2. HAVING

In [ ]:
# 10건 이상 처방된 약물
q("""
SELECT drug_name, COUNT(*) AS 처방건수
FROM prescription GROUP BY drug_name
HAVING COUNT(*) >= 10 ORDER BY 처방건수 DESC
""")

## 3. Table 1 맛보기

In [ ]:
# 성별 × 연령군 분포
q("""
SELECT sex,
  CASE WHEN 2026 - CAST(substr(birth_date,1,4) AS INT) >= 65
       THEN '65세 이상' ELSE '65세 미만' END AS 연령군,
  COUNT(*) AS 명수
FROM patient GROUP BY sex, 연령군 ORDER BY sex, 연령군
""")

## 연습문제

In [ ]:
# [연습 Q1] KCD 코드별 진단 건수를 많은 순으로
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 검사 종류별 평균 결과값과 검사 건수는?
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 5회 이상 내원한 환자의 ID와 내원 횟수는?
# 힌트: GROUP BY patient_id + HAVING
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 혈액형별 환자 수 분포는?
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q5] 연도별 입원 건수 추이는?
# 힌트: strftime('%Y', …)
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. KCD별 진단 건수**
```sql
SELECT kcd_code, dx_name, COUNT(*) AS n FROM diagnosis GROUP BY kcd_code, dx_name ORDER BY n DESC;
```

**A2. 검사별 평균**
```sql
-- COUNT(*)와 COUNT(result_value)를 함께 — 평균의 분모는 후자다
SELECT test_name, ROUND(AVG(result_value),2) AS 평균,
       COUNT(*) AS 전체건수, COUNT(result_value) AS 값있는건수
FROM lab GROUP BY test_name;
```

**A3. 5회 이상 내원**
```sql
SELECT patient_id, COUNT(*) AS n FROM visit GROUP BY patient_id HAVING COUNT(*) >= 5 ORDER BY n DESC;
```

**A4. 혈액형 분포**
```sql
SELECT blood_type, COUNT(*) FROM patient GROUP BY blood_type ORDER BY 2 DESC;
```

**A5. 연도별 입원**
```sql
SELECT strftime('%Y', visit_date) AS 연도, COUNT(*) FROM visit WHERE visit_type='입원' GROUP BY 연도 ORDER BY 연도;
```